In [7]:
import surprise
import pandas as pd
import numpy as np
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold

In [9]:
ratings=pd.read_csv("D:\\AshleshaRuchika\\PGCP-AI\Machine Learning\\Cases_Rec_Sys\\ml-100k\\u.data",sep="\t",names=['uid','iid','rating','ts'])
print(ratings.head())
ratings.drop('ts',axis=1,inplace=True)

   uid  iid  rating         ts
0  196  242       3  881250949
1  186  302       3  891717742
2   22  377       1  878887116
3  244   51       2  880606923
4  166  346       1  886397596


In [10]:
lowest_rating=ratings["rating"].min()
highest_rating=ratings["rating"].max()
lowest_rating,highest_rating

(1, 5)

In [11]:
reader=surprise.Reader(rating_scale=(lowest_rating,highest_rating))
data=surprise.Dataset.load_from_df(ratings,reader)   

In [12]:
similarity_options={'name':'cosine','user_based':True}
algo=surprise.KNNBasic(sim_options=similarity_options)
output=algo.fit(data.build_full_trainset())

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [13]:
param_grid={"k":[20,30,50,70,90],"user_based":[True]}
kFold=KFold(n_splits=5,random_state=26,shuffle=True)
gs=GridSearchCV(surprise.KNNBasic,param_grid,measures=["rmse","mae"],cv=kFold)
gs.fit(data)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [14]:
print(gs.best_score["rmse"])
print(gs.best_params["mae"])

0.9765179159493386
{'k': 20, 'user_based': True}


In [21]:
iids=ratings['iid'].unique()
print(iids)

[ 242  302  377 ... 1637 1630 1641]


In [22]:
user=50
u_iid=ratings[ratings['uid']==user]['iid'].unique()
print("List of item rated by user: ",u_iid)
print("No of items rated by user{0}: {1}".format(user,len(u_iid)))
iids_to_predict=np.setdiff1d(iids,u_iid)
print("Items not rated by user or those items for which the expected ratings are to be predicted ",iids_to_predict)

List of item rated by user:  [ 246  823  253  475 1084  286    9  125  123  325  508  288  319  324
  276 1008 1010  268  544   15  327  124  547  100]
No of items rated by user50: 24
Items not rated by user or those items for which the expected ratings are to be predicted  [   1    2    3 ... 1680 1681 1682]


In [23]:
testSet=[[user,iid,0.] for iid in iids_to_predict]
predictions=algo.test(testSet)
exp_ratings=[(predictions[i].iid,predictions[i].est) for i in range(0,len(predictions))]
exp_ratings=pd.DataFrame(exp_ratings,columns=['iid','est_ratings'])

In [24]:
movies=pd.read_csv("D:\\AshleshaRuchika\\PGCP-AI\\Machine Learning\\Cases_Rec_Sys\\ml-100k\\movies_list.csv",encoding='latin-1')
movies.columns

Index(['movie id ', ' movie title ', ' release date ', ' video release date ',
       'IMDb URL ', ' unknown ', ' Action ', ' Adventure ', ' Animation ',
       'Children's ', ' Comedy ', ' Crime ', ' Documentary ', ' Drama ',
       ' Fantasy ', 'Film-Noir ', ' Horror ', ' Musical ', ' Mystery ',
       ' Romance ', ' Sci-Fi ', 'Thriller ', ' War ', ' Western'],
      dtype='str')

In [30]:
user=50
u_iid=ratings[ratings['uid']==user]['iid'].unique()
iids_to_predict=np.setdiff1d(iids,u_iid)
testset=[[user,iid,0.] for iid in iids_to_predict]
predictions=algo.test(testSet)
exp_ratings=[(predictions[i].iid,predictions[i].est) for i in range(0,len(predictions))]
exp_ratings=pd.DataFrame(exp_ratings,columns=['iid','est_ratings'])
exp_ratings=exp_ratings.merge(movies[['movie id ',' movie title ',' release date ']],
                             how='left',left_on='iid',
                             right_on='movie id ')
exp_ratings.sort_values(by='est_ratings',ascending=False).head(10)


,iid,est_ratings,movie id,movie title,release date
1475,1500,5.0,1500,Santa with Muscles (1996),08-Nov-96
1442,1467,5.0,1467,"Saint of Fort Washington, The (1993)",01-Jan-93
1164,1189,5.0,1189,Prefontaine (1997),24-Jan-97
1097,1122,5.0,1122,They Made Me a Criminal (1939),01-Jan-39
1628,1653,5.0,1653,Entertaining Angels: The Dorothy Day Story (1996),27-Sep-96
793,814,5.0,814,"Great Day in Harlem, A (1994)",01-Jan-94
1511,1536,5.0,1536,Aiqing wansui (1994),22-Jul-96
1268,1293,5.0,1293,Star Kid (1997),16-Jan-98
1574,1599,5.0,1599,Someone Else's America (1995),10-May-96
1176,1201,5.0,1201,Marlene Dietrich: Shadow and Light (1996),02-Apr-96
